# 03 — Prepare

**billboard-hot100** — shape the two interim clean tables into the
**chart-ready / analysis-ready** datasets `04-viz` will consume, and export
the sellable package. This stage has TWO jobs:

1. **Build the chart data** — the shaped, aggregated per-chart tables each
   chart draws from (genre presence + rank per year / per decade). The project's
   lead chart is **genre rank over time (a bump chart)**, so the core table here
   is the per-(year, genre) appearances / share / rank table the bump ranks from.
2. **Export the sellable package** — CSV + Excel + Parquet + a codebook.

> ⭐ **03 OWNS the chart data; 04-viz only CONSUMES it.** The join
> (`hot100_clean` → `artist_genre`) and ALL aggregation live here, never in
> `04-viz`. `04-viz` reads the `chart_*` tables below to explore framing; if
> exploration reveals a new cut is needed, the fix is to come BACK here, amend
> this notebook, and re-run — not to build chart tables inside `04`.

All heavy work is **DuckDB SQL** (workspace rule). The genre buckets and the
top-tag-wins rule are taken EXACTLY as `02-clean` produced them — nothing is
re-bucketed here.

In [ ]:
import sys, os
from pathlib import Path

# Navigate to project root regardless of how the kernel was launched
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 120)

from src.ingest import load_config
from src.clean_quality import (
    get_connection, run_sql, save_processed, register_source, get_sources,
)
from src.prepare import package_dataset

cfg = load_config('config.yaml')
con = get_connection(cfg)
print('hot100_clean :', con.execute('SELECT COUNT(*) FROM hot100_clean').fetchone()[0], 'rows')
print('artist_genre :', con.execute('SELECT COUNT(*) FROM artist_genre').fetchone()[0], 'rows')

## Step 1 — `chart_genre_by_year` (the core table)

One row per **(year, genre)**. For each year 1958–2026 and each of the 12
aggregate genres we compute:

- **`appearances`** — COUNT of `hot100_clean` weekly chart slots held that year
  by artists mapping to that genre.
- **`share`** — `appearances` as a **0–100 percent** of that year's TOTAL chart
  slots (so each year sums to ~100 across all 12 genres).
- **`rank`** — the genre's **dense rank within that year** by `appearances`
  (1 = most appearances). This is the per-year rank the bump chart draws.

**How genre is attached:** LEFT JOIN `hot100_clean` → `artist_genre` on the exact
Billboard artist spelling, and **COALESCE a null genre to `Other/Unknown`**.
Appearances by artists below the ≥ 20-appearance enrichment threshold have NO
`artist_genre` row, so they fall into `Other/Unknown` too — the documented
coverage residual. `Other/Unknown` is kept IN this table for honesty and the
export (it is excluded only from the headline rank race in Step 3).

This single table supports both the **count view** and the **share view**, and
carries the per-year `rank` the bump needs — so `04-viz` never re-derives it.

In [ ]:
chart_genre_by_year = run_sql('''
    WITH tagged AS (
        SELECT h.year,
               COALESCE(ag.genre, 'Other/Unknown') AS genre
        FROM hot100_clean h
        LEFT JOIN artist_genre ag ON h.artist = ag.artist
    ),
    by_year AS (
        SELECT year, genre, COUNT(*) AS appearances
        FROM tagged GROUP BY year, genre
    ),
    year_totals AS (
        SELECT year, SUM(appearances) AS total FROM by_year GROUP BY year
    )
    SELECT b.year,
           b.genre,
           b.appearances,
           ROUND(100.0 * b.appearances / t.total, 2) AS share,
           DENSE_RANK() OVER (PARTITION BY b.year ORDER BY b.appearances DESC) AS rank
    FROM by_year b
    JOIN year_totals t USING (year)
    ORDER BY b.year, b.appearances DESC
''', con)

con.execute('DROP TABLE IF EXISTS chart_genre_by_year')
con.execute('CREATE TABLE chart_genre_by_year AS SELECT * FROM chart_genre_by_year')
save_processed(chart_genre_by_year, cfg, 'chart_genre_by_year.parquet')
print('chart_genre_by_year:', len(chart_genre_by_year), 'rows '
      f'(≈12 genres × 69 years, minus genre-years with zero appearances)')
chart_genre_by_year.head(12)

### Verify — each year's shares sum to ~100
A quick invariant: the per-year `share` column should total 100 (±rounding) for
every year, confirming the denominator is that year's full chart-slot count.

In [ ]:
share_check = run_sql('''
    SELECT MIN(s) AS min_year_share_sum, MAX(s) AS max_year_share_sum
    FROM (SELECT year, SUM(share) AS s FROM chart_genre_by_year GROUP BY year)
''', con).iloc[0]
print(f'per-year share sum ranges {share_check["min_year_share_sum"]:.1f} '
      f'→ {share_check["max_year_share_sum"]:.1f} (should be ≈100)')

## Step 2 — `chart_genre_by_decade` (smoother alternative cut)

Decade-grain companion to Step 1 — same columns (`appearances`, `share`,
`rank`), grouped by `decade` (the decade's first year, already on
`hot100_clean`). A smoother, less jumpy cut `04-viz` can use for an alternative
bump / area view. Same LEFT JOIN + COALESCE logic; `rank` is the dense rank of
the genre **within that decade**.

In [ ]:
chart_genre_by_decade = run_sql('''
    WITH tagged AS (
        SELECT h.decade,
               COALESCE(ag.genre, 'Other/Unknown') AS genre
        FROM hot100_clean h
        LEFT JOIN artist_genre ag ON h.artist = ag.artist
    ),
    by_decade AS (
        SELECT decade, genre, COUNT(*) AS appearances
        FROM tagged GROUP BY decade, genre
    ),
    decade_totals AS (
        SELECT decade, SUM(appearances) AS total FROM by_decade GROUP BY decade
    )
    SELECT b.decade,
           b.genre,
           b.appearances,
           ROUND(100.0 * b.appearances / t.total, 2) AS share,
           DENSE_RANK() OVER (PARTITION BY b.decade ORDER BY b.appearances DESC) AS rank
    FROM by_decade b
    JOIN decade_totals t USING (decade)
    ORDER BY b.decade, b.appearances DESC
''', con)

con.execute('DROP TABLE IF EXISTS chart_genre_by_decade')
con.execute('CREATE TABLE chart_genre_by_decade AS SELECT * FROM chart_genre_by_decade')
save_processed(chart_genre_by_decade, cfg, 'chart_genre_by_decade.parquet')
print('chart_genre_by_decade:', len(chart_genre_by_decade), 'rows')
chart_genre_by_decade.head(12)

## Step 3 — `chart_genre_rank_top` (the readable ~6-line bump set)

A convenience view of `chart_genre_by_year` RESTRICTED to the genres that
actually compete for the top of the rank race, so `04-viz` can draw a readable
~6-line bump without re-deriving the set.

**Rule used (decided empirically, documented): the top-N genres by TOTAL
appearances across all years, N = 6, EXCLUDING `Other/Unknown`.**

We considered two candidate rules:
- *Rule A — genres that ever reach the top-6 rank in any single year.* On this
  data that admits **all 11 real genres** (even Blues/Jazz/Folk top the sparse
  early years), which is far too many lines for a readable bump.
- *Rule B — the 6 genres with the highest total appearances across all
  years (chosen).* This yields a clean, readable headline set and matches what
  the lead chart is about: the genres that genuinely dominate the Hot 100.

`Other/Unknown` is deliberately kept OUT of this headline race — it is a
coverage residual (unmatched artists + the < 20-appearance tail), not a genre —
but it REMAINS in `chart_genre_by_year` for honesty and the export. The chosen
top-6 set is printed below.

In [ ]:
TOP_N = 6
top_set = run_sql(f'''
    SELECT genre, SUM(appearances) AS total_appearances
    FROM chart_genre_by_year
    WHERE genre <> 'Other/Unknown'
    GROUP BY genre
    ORDER BY total_appearances DESC
    LIMIT {TOP_N}
''', con)
top_genres = top_set['genre'].tolist()
print(f'top-{TOP_N} genres by total appearances (the bump race):')
display(top_set)
print('top set →', top_genres)

In [ ]:
# Build the restricted view as a materialized table (parameterized by top_genres).
placeholders = ', '.join(f"'{g}'" for g in top_genres)
chart_genre_rank_top = run_sql(f'''
    SELECT year, genre, appearances, share,
           DENSE_RANK() OVER (PARTITION BY year ORDER BY appearances DESC) AS rank
    FROM chart_genre_by_year
    WHERE genre IN ({placeholders})
    ORDER BY year, appearances DESC
''', con)
# NOTE: rank here is re-computed WITHIN the top set so it is a dense 1..N race
# with no gaps from the excluded genres — exactly what the bump chart plots.
con.execute('DROP TABLE IF EXISTS chart_genre_rank_top')
con.execute('CREATE TABLE chart_genre_rank_top AS SELECT * FROM chart_genre_rank_top')
save_processed(chart_genre_rank_top, cfg, 'chart_genre_rank_top.parquet')
print('chart_genre_rank_top:', len(chart_genre_rank_top), 'rows',
      f'({len(top_genres)} genres × {chart_genre_rank_top["year"].nunique()} years)')
chart_genre_rank_top.head(12)

## Step 4 — supporting shaped table: `chart_genre_number_ones_by_decade`

A cheap secondary angle for `04-viz`: per-genre count of **genre-#1 years** by
decade — i.e. how many years in each decade a genre held `rank == 1` (the most
chart slots that year) in `chart_genre_by_year`. It summarizes which genre
*owned* each decade and is a clean one-query add (no over-building). Kept at
decade grain so it reads as a compact matrix.

In [ ]:
chart_genre_number_ones_by_decade = run_sql('''
    SELECT decade, genre, n_top_years
    FROM (
        SELECT CAST(FLOOR(year / 10) * 10 AS INTEGER) AS decade,
               genre,
               COUNT(*) AS n_top_years
        FROM chart_genre_by_year
        WHERE rank = 1
        GROUP BY 1, 2
    )
    ORDER BY decade, n_top_years DESC
''', con)
con.execute('DROP TABLE IF EXISTS chart_genre_number_ones_by_decade')
con.execute('CREATE TABLE chart_genre_number_ones_by_decade AS '
            'SELECT * FROM chart_genre_number_ones_by_decade')
save_processed(chart_genre_number_ones_by_decade, cfg,
               'chart_genre_number_ones_by_decade.parquet')
print('chart_genre_number_ones_by_decade:', len(chart_genre_number_ones_by_decade), 'rows')
chart_genre_number_ones_by_decade

## Step 4c — song-level `chart_songs_pre_streaming` / `chart_songs_streaming_era`

The two song-grain tables behind the social **longest-charting-songs** charts
(06-viz-social charts 3 & 4). A **song** is `(song, artist)`; its **weeks on
chart** = `MAX(weeks_on_chart)` and its **era** = `MIN(year)` (first chart year).
Genre is joined from `artist_genre` by artist (COALESCE null → Other/Unknown).
We split at the **2013 streaming break** and take the **top 10 songs by weeks**
in each era:

- `chart_songs_pre_streaming` — first chart year **1958–2012**.
- `chart_songs_streaming_era` — first chart year **2013–2026**.

This is **03's** job (shaped, aggregated chart-ready data) — `06` only consumes
these tables and adds presentation (the shared genre→color map, the `N wks` label).

> **Streaming-break caveat (carried into the chart + SOURCES):** the streaming
> ramp bled into the late-2000s/early-2010s, so the pre-streaming top is dominated
> by 2008–2012 songs plus perennial Christmas songs. The honest contrast still
> holds — pre-streaming max ~87 weeks vs ~112 in the streaming era, and the genre
> mix flips from Pop/Rock/Country toward R&B-Soul / Dance-Electronic / Pop.

In [ ]:
con.execute('''
    CREATE OR REPLACE TEMP VIEW _songs AS
    SELECT h.song, h.artist,
           MAX(h.weeks_on_chart) AS weeks,
           MIN(h.year)           AS first_year,
           ANY_VALUE(g.genre)    AS genre
    FROM hot100_clean h
    LEFT JOIN artist_genre g ON g.artist = h.artist
    GROUP BY h.song, h.artist
''')
for tbl, lo, hi in [('chart_songs_pre_streaming', 1958, 2012),
                    ('chart_songs_streaming_era', 2013, 2026)]:
    df = run_sql(f'''
        SELECT song, artist, COALESCE(genre, 'Other/Unknown') AS genre, weeks
        FROM _songs
        WHERE first_year BETWEEN {lo} AND {hi}
        ORDER BY weeks DESC, song
        LIMIT 10
    ''', con)
    con.execute(f'DROP TABLE IF EXISTS {tbl}')
    con.register('_tmp_df', df)
    con.execute(f'CREATE TABLE {tbl} AS SELECT * FROM _tmp_df')
    con.unregister('_tmp_df')
    save_processed(df, cfg, f'{tbl}.parquet')
    print(f'{tbl}: {len(df)} rows  (first-year {lo}-{hi}; top weeks = {int(df["weeks"].max())})')
chart_songs_pre_streaming = run_sql('SELECT * FROM chart_songs_pre_streaming ORDER BY weeks DESC, song', con)
chart_songs_streaming_era = run_sql('SELECT * FROM chart_songs_streaming_era ORDER BY weeks DESC, song', con)
display(chart_songs_pre_streaming)
display(chart_songs_streaming_era)

## Step 5 — verification / sanity checks (printed inline)

Four checks the task asks for: the core table row count + sample, the #1 genre
per decade (does the rank-over-time story read sensibly?), the chosen top set +
rule, and confirmation that **Metal** and **Latin** appear with non-zero
appearances in the years they charted.

### 5a. `chart_genre_by_year` row count + sample

In [ ]:
n_core = con.execute('SELECT COUNT(*) FROM chart_genre_by_year').fetchone()[0]
n_years = con.execute('SELECT COUNT(DISTINCT year) FROM chart_genre_by_year').fetchone()[0]
n_genres = con.execute('SELECT COUNT(DISTINCT genre) FROM chart_genre_by_year').fetchone()[0]
print(f'chart_genre_by_year: {n_core} rows  '
      f'({n_genres} genres × {n_years} years, minus zero-appearance genre-years)')
display(run_sql('SELECT * FROM chart_genre_by_year ORDER BY year, rank LIMIT 15', con))

### 5b. #1 genre per DECADE — does the rank-over-time story read sensibly?

The genre with the most chart slots in each decade (from
`chart_genre_by_decade`, `rank == 1`). We expect Rock / R&B-Soul dominant
mid-century and Hip-Hop/Rap rising late — a sanity check that the bump story
is believable before `04-viz` draws it.

In [ ]:
top_by_decade = run_sql('''
    SELECT decade, genre AS top_genre, appearances, share, rank
    FROM chart_genre_by_decade
    WHERE rank = 1
    ORDER BY decade
''', con)
display(top_by_decade)

# Also the per-YEAR #1 (collapsed to runs) for a finer look at the hand-off.
print('\n#1 genre each YEAR (year : genre):')
yr1 = run_sql('''
    SELECT year, genre FROM chart_genre_by_year WHERE rank = 1 ORDER BY year
''', con)
print('  ' + '  '.join(f'{int(r.year)}:{r.genre}' for r in yr1.itertuples()))

### 5c. Top set chosen for `chart_genre_rank_top` + the rule

In [ ]:
print(f'Rule: top-{TOP_N} genres by TOTAL appearances across all years, '
      'excluding Other/Unknown.')
print(f'Chosen top set: {top_genres}')
display(top_set)

### 5d. Metal & Latin appear with non-zero appearances in the years they charted
The owner specifically wanted a Metal bucket (and Latin was broken out too), so
confirm both are present in the data with real counts — first/last charting year,
number of years present, total appearances, and the peak single-year count.

In [ ]:
ml = run_sql('''
    SELECT genre,
           MIN(year) AS first_year,
           MAX(year) AS last_year,
           COUNT(*)  AS years_present,
           SUM(appearances) AS total_appearances,
           MAX(appearances) AS peak_year_appearances
    FROM chart_genre_by_year
    WHERE genre IN ('Metal', 'Latin') AND appearances > 0
    GROUP BY genre ORDER BY genre
''', con)
display(ml)
assert len(ml) == 2 and (ml['total_appearances'] > 0).all(), \
    'Metal and/or Latin missing or zero — expected both present with non-zero appearances'
print('✓ Metal and Latin both present with non-zero appearances')
print('\nMetal — top charting years:')
display(run_sql("SELECT year, appearances, share, rank FROM chart_genre_by_year "
                "WHERE genre='Metal' ORDER BY appearances DESC LIMIT 5", con))
print('Latin — top charting years:')
display(run_sql("SELECT year, appearances, share, rank FROM chart_genre_by_year "
                "WHERE genre='Latin' ORDER BY appearances DESC LIMIT 5", con))

## Step 6 — export the sellable package `billboard_hot100_genre_v1`

Job 2 of this stage. The export's **main** data is the analysis-ready
genre-by-year table (`year, genre, appearances, share, rank`) — the thing a
buyer wants. We also ship the **per-artist genre mapping** (`artist, genre,
total_appearances`) as a second file, since that artist→genre enrichment is the
real work behind the dataset.

Formats come from `config.export.formats` (**CSV + Excel + Parquet**) via the
project's `package_dataset()` helper, which also writes the REQUIRED codebook.

**PII:** `config.export.strip_pii_columns` is empty — this is public Hot 100
chart data plus a crowd-sourced genre mapping, so there is nothing to strip.
`package_dataset()` calls `strip_pii()` regardless (a no-op here); noted for the
record.

In [ ]:
# Main export frame = the analysis-ready genre-by-year table.
export_main = con.execute(
    'SELECT year, genre, appearances, share, rank '
    'FROM chart_genre_by_year ORDER BY year, rank'
).df()

genre_year_codebook = {
    'year': 'Calendar year of the Billboard Hot 100 weekly charts (1958–2026).',
    'genre': ('Aggregate genre bucket (one of 12: Pop, Rock, Hip-Hop/Rap, '
              'R&B/Soul, Country, Dance/Electronic, Latin, Folk, Jazz, Blues, '
              'Metal, Other/Unknown), assigned per ARTIST from the artist\'s '
              'single highest-count non-noise MusicBrainz tag (top-tag-wins).'),
    'appearances': ('Count of weekly Hot 100 chart slots held that year by '
                    'artists of that genre (one slot = one song on one weekly '
                    'chart; a song in the Top 100 for 10 weeks counts 10).'),
    'share': ("The genre's appearances as a percent (0–100) of that year's "
              'total chart slots; all genres in a year sum to ~100.'),
    'rank': ("The genre's rank that year by appearances (dense rank, 1 = most "
             'appearances). Ties share a rank; the next rank is not skipped.'),
}

export_notes = '''
    billboard-hot100 — genre rank over time (v1)

    WHAT THIS IS
    Genre presence on the Billboard Hot 100, 1958–2026, as yearly counts, shares,
    and ranks across 12 aggregate genres. Built to answer: how have music genres
    risen and fallen at the top of the Hot 100 over the decades?

    HOW GENRE IS ASSIGNED (and its limits)
    Billboard carries no genre. Genre is derived at the ARTIST level from
    MusicBrainz crowd-sourced tags: each artist is bucketed by its single
    highest-MB-count non-noise tag ("top tag wins"). An artist's WHOLE catalog
    therefore counts as that one genre for every chart week — a song is not
    genre-tagged individually. Crossover superstars whose top MB tag is "pop"
    (e.g. Michael Jackson, Beyoncé, Taylor Swift) read as Pop, so Pop runs larger
    and the roots genres smaller than a lineage-based taxonomy would.

    OTHER/UNKNOWN CAVEAT
    Other/Unknown = chart slots whose artist has no usable genre, which is the sum
    of (a) artists MusicBrainz matched but with no genre-bearing tag and (b) the
    long tail of artists with fewer than 20 Hot 100 appearances, who were below
    the enrichment threshold and never genre-looked-up. It is a coverage residual,
    not a musical category; it is kept in the data for honesty but excluded from
    the headline genre rank race.

    SERIES BREAK
    Billboard's Hot 100 methodology changed over the decades (sales + radio
    airplay historically; digital downloads folded in 2005, streaming from 2007
    and progressively weighted through the 2010s), so chart presence is not
    measured identically across eras.

    PII: none. Public chart data + a crowd-sourced artist→genre mapping.

    Sources: Billboard Hot 100 full chart history (mhollingshead/billboard-hot-100,
    GitHub; underlying chart © Billboard, non-commercial research use) and
    MusicBrainz Web Service v2 artist tags (CC0 / CC-BY-NC-SA). See SOURCES.md.
'''

written_main = package_dataset(
    export_main, cfg, name='billboard_hot100_genre_v1',
    codebook=genre_year_codebook, notes=export_notes,
)

### Second file — the per-artist genre mapping (`..._artist_genre`)
The enrichment work: one row per enriched artist with its aggregate genre and
total Hot 100 appearances. Shipped alongside the main dataset (same formats +
its own codebook) so a buyer can see exactly how each artist was classified.

In [ ]:
export_artist = con.execute(
    'SELECT artist, genre, total_appearances, matched, top_tags '
    'FROM artist_genre ORDER BY total_appearances DESC'
).df()

artist_codebook = {
    'artist': 'Artist name as credited on the Billboard Hot 100 (the join key).',
    'genre': ('Aggregate genre bucket assigned to the artist by top-tag-wins '
              '(see the main dataset codebook for the full rule and 12-bucket set).'),
    'total_appearances': ("The artist's total weekly Hot 100 chart slots across "
                          'all years 1958–2026 (used as the ≥ 20 enrichment '
                          'threshold at ingest).'),
    'matched': ('TRUE if MusicBrainz returned an artist match for the lookup; '
                'FALSE if unmatched (then genre = Other/Unknown).'),
    'top_tags': ("The artist's raw MusicBrainz tag names, comma-joined, highest "
                 'MB count first — kept verbatim for auditability.'),
}

written_artist = package_dataset(
    export_artist, cfg, name='billboard_hot100_genre_v1_artist_genre',
    codebook=artist_codebook,
    notes=('Per-artist genre mapping behind billboard_hot100_genre_v1. Genre is '
           'artist-level from MusicBrainz top-tag-wins; an artist\'s whole catalog '
           'counts as one genre. Only artists with >= 20 Hot 100 appearances were '
           'enriched; see the main codebook for the Other/Unknown caveat. PII: none.'),
)

### Third/fourth files — the per-era longest-charting songs
The two song-grain tables behind social charts 3 & 4, shipped so a buyer gets the
exact top-10-by-era data the charts plot (each as CSV/Excel/Parquet + codebook).

In [ ]:
_song_codebook = {
    'song': 'Song title as credited on the Billboard Hot 100.',
    'artist': 'Artist as credited on the Hot 100 (the genre join key).',
    'genre': ('Aggregate genre of the credited artist (top-tag-wins via MusicBrainz; '
              'see the main dataset codebook). Null -> Other/Unknown.'),
    'weeks': "The song's total weeks on the Hot 100 (MAX weeks_on_chart across its run).",
}
_song_note_pre = (
    'Top 10 Hot 100 songs by total weeks on chart whose FIRST chart year is 1958-2012 '
    '(pre-streaming era). Era = MIN(year) for the (song, artist). CAVEAT: the streaming '
    'ramp bled into 2008-2012, so this top is dominated by 2008-2012 songs + perennial '
    'Christmas songs. See SOURCES.md (2013 streaming break). PII: none.'
)
_song_note_stream = (
    'Top 10 Hot 100 songs by total weeks on chart whose FIRST chart year is 2013-2026 '
    '(streaming era, after on-demand streaming was folded into the formula). Era = '
    'MIN(year) for the (song, artist). See SOURCES.md (2013 streaming break). PII: none.'
)
written_songs_pre = package_dataset(
    chart_songs_pre_streaming, cfg, name='billboard_hot100_longest_charting_pre_streaming',
    codebook=_song_codebook, notes=_song_note_pre,
)
written_songs_stream = package_dataset(
    chart_songs_streaming_era, cfg, name='billboard_hot100_longest_charting_streaming_era',
    codebook=_song_codebook, notes=_song_note_stream,
)

### Confirm export files written + their row counts

In [ ]:
from pathlib import Path as _P
print('=== export files written ===')
for label, written, df in [('main (genre-by-year)', written_main, export_main),
                           ('artist mapping', written_artist, export_artist),
                           ('songs pre-streaming', written_songs_pre, chart_songs_pre_streaming),
                           ('songs streaming-era', written_songs_stream, chart_songs_streaming_era)]:
    print(f'\n{label}  —  {len(df):,} rows × {len(df.columns)} cols')
    for fmt, path in written.items():
        p = _P(path)
        print(f'  {fmt:9s} {"OK" if p.exists() else "MISSING":7s} '
              f'{p.stat().st_size:>9,} bytes  {p}')

## Step 7 — provenance

Register the processed chart tables + the export in `_sources`, consistent with
how `01`/`02` recorded their tables. All of these derive from `hot100_clean` ⋈
`artist_genre`, so they inherit the same methodology notes and the Hot 100
series break.

In [ ]:
_prep_methodology = (
    '03-prepare DuckDB SQL: LEFT JOIN hot100_clean -> artist_genre on the Billboard '
    'artist spelling, COALESCE null genre to Other/Unknown, GROUP BY year (or decade) '
    'and genre to get appearances, share (% of that period total chart slots), and a '
    'dense rank by appearances. chart_genre_rank_top restricts to the top-6 genres by '
    'total appearances (excl. Other/Unknown). No re-bucketing — genres are exactly as '
    '02-clean produced them (top-tag-wins).'
)
_prep_breaks = (
    'Inherits the Hot 100 methodology break: sales + airplay historically, digital '
    'downloads 2005, streaming 2007+, and the load-bearing 2013 break when on-demand/'
    'YouTube streaming was folded into the formula (further video/bundle changes '
    '~2018-2020). Genre presence is not measured identically across eras -- the '
    'post-2013 Hip-Hop/Rap + Pop surge and longer chart runs are partly a streaming '
    'artifact. Other/Unknown is a coverage residual (unmatched artists + the '
    '<20-appearance tail), not a genre.'
)
for tbl, desc in [
    ('chart_genre_by_year',
     'Per-(year, genre) Hot 100 appearances, share %, and dense rank (1958-2026).'),
    ('chart_genre_by_decade',
     'Per-(decade, genre) Hot 100 appearances, share %, and dense rank.'),
    ('chart_genre_rank_top',
     'chart_genre_by_year restricted to the top-6 genres by total appearances '
     '(the readable bump race; rank re-densed within the set).'),
    ('chart_genre_number_ones_by_decade',
     'Per-(decade, genre) count of years the genre held the #1 (most-appearances) '
     'spot — which genre owned each decade.'),
    ('chart_songs_pre_streaming',
     'Top 10 Hot 100 songs by total weeks on chart with first chart year 1958-2012 '
     '(pre-streaming era); song, artist, genre, weeks.'),
    ('chart_songs_streaming_era',
     'Top 10 Hot 100 songs by total weeks on chart with first chart year 2013-2026 '
     '(streaming era); song, artist, genre, weeks.'),
]:
    register_source(
        con, tbl, 'billboard-hot100 prepared chart data (03-prepare)',
        url=cfg['sources']['hot100_all']['url'],
        license=cfg['sources']['hot100_all']['license'],
        notes=desc, methodology=_prep_methodology, series_breaks=_prep_breaks,
    )

register_source(
    con, 'billboard_hot100_genre_v1',
    'billboard-hot100 sellable export (03-prepare)',
    url=cfg['sources']['hot100_all']['url'],
    license=cfg['sources']['hot100_all']['license'],
    notes=('Export package in export/: main genre-by-year dataset (year, genre, '
           'appearances, share, rank) + a per-artist genre-mapping file, each as '
           'CSV/Excel/Parquet with a codebook. Public chart data + crowd-sourced '
           'genre mapping; no PII.'),
    methodology=_prep_methodology, series_breaks=_prep_breaks,
)
display(get_sources(con)[['duckdb_table', 'source_name', 'retrieved']])

---
**Next:** `04-viz.ipynb` — CONSUME these `chart_*` tables (read-only) to explore
the genre-rank-over-time framing and decide the story. If exploration reveals a
needed new cut, amend THIS notebook and re-run — `04` must not build chart tables.

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released.

In [ ]:
con.close()
print('connection closed')